<h1 style="text-align:center">Recurrent Neural Network</h1>                                                       


## Part 1: Generate text 


In [124]:
shakespeare = open("shakespeare.txt").read()
print(shakespeare[:250])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [125]:
unique_characters = sorted(set(shakespeare))
vocab = sorted(set(shakespeare))
print(len(unique_characters))
print(shakespeare[:10])

65
First Citi


In [126]:
char2idx = {char: index for index, char in enumerate(unique_characters)}

In [127]:
import numpy as np
print('{')
for char, _ in zip(char2idx, range(12)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')


{
  '\n':   0,
  ' ' :   1,
  '!' :   2,
  '$' :   3,
  '&' :   4,
  "'" :   5,
  ',' :   6,
  '-' :   7,
  '.' :   8,
  '3' :   9,
  ':' :  10,
  ';' :  11,
  ...
}


In [128]:
text_as_int = [char2idx[char] for char in shakespeare]
print("characters mapped (char2index) : ")
print('{')
for index in zip(text_as_int, range(10)):
    print(f'  {index[0]} : {shakespeare[index[1]]},')
print('  ...\n}')  

characters mapped (char2index) : 
{
  18 : F,
  47 : i,
  56 : r,
  57 : s,
  58 : t,
  1 :  ,
  15 : C,
  47 : i,
  58 : t,
  47 : i,
  ...
}


In [129]:
import tensorflow as tf
seq_length = 100
char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)

In [130]:
idx2char = np.array(vocab)
for i in char_dataset.take(5):
     print(idx2char[i.numpy()])

F
i
r
s
t


In [131]:
sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

In [132]:
print(sequences.take(1))
for item in sequences.take(2):
    print(repr(item.numpy()))
    print(repr(''.join(idx2char[item.numpy()])))

<_TakeDataset element_spec=TensorSpec(shape=(101,), dtype=tf.int32, name=None)>
array([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43,
       44, 53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39,
       52, 63,  1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1,
       51, 43,  1, 57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31,
       54, 43, 39, 49,  6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56,
       57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1],
      dtype=int32)
'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '
array([39, 56, 43,  1, 39, 50, 50,  1, 56, 43, 57, 53, 50, 60, 43, 42,  1,
       56, 39, 58, 46, 43, 56,  1, 58, 53,  1, 42, 47, 43,  1, 58, 46, 39,
       52,  1, 58, 53,  1, 44, 39, 51, 47, 57, 46, 12,  0,  0, 13, 50, 50,
       10,  0, 30, 43, 57, 53, 50, 60, 43, 42,  8,  1, 56, 43, 57, 53, 50,
       60, 43, 42,  8,  0,  0, 18, 47, 56, 

In [133]:
dataset = sequences.map(lambda x: (x[:-1], x[1:]))

In [134]:

for input_example, target_example in  dataset.take(1):
    print ('Input data: ', repr(''.join(idx2char[input_example.numpy()])))
    print ('Target data:', repr(''.join(idx2char[target_example.numpy()])))


Input data:  'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou'
Target data: 'irst Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou '


In [135]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

## Model: RNN

In [136]:
vocab_size = len(vocab)
def build_model(batch_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Embedding(vocab_size, 256, input_length=batch_size),
        tf.keras.layers.RNN(tf.keras.layers.GRUCell(512), return_sequences=True, stateful=True),
        tf.keras.layers.Dense(vocab_size, activation='softmax')
    ])
    return model

In [137]:
model = build_model(BATCH_SIZE)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy())
model.summary()

/opt/miniconda3/envs/venvMacOS/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_3 (RNN)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [138]:
history = model.fit(dataset, epochs=30)

Epoch 1/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 67s 380ms/step - loss: 2.3724
Epoch 2/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 65s 373ms/step - loss: 1.8244
Epoch 3/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 66s 380ms/step - loss: 1.6240
Epoch 4/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 73s 423ms/step - loss: 1.5201
Epoch 5/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 72s 412ms/step - loss: 1.4576
Epoch 6/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 74s 427ms/step - loss: 1.4146
Epoch 7/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 71s 408ms/step - loss: 1.3828
Epoch 8/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 71s 409ms/step - loss: 1.3567
Epoch 9/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 72s 414ms/step - loss: 1.3344
Epoch 10/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 71s 411ms/step - loss: 1.3157
Epoch 11/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 71s 408ms/step - loss: 1.2989
Epoch 12/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 70s 404ms/step - loss: 1.2826
Epoch 13/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 68s 390ms/step - loss: 1.2690
Epoch 14/30
172/172 ━━━━━━━━━━━━━━━━━━━━ 67s 384ms/step - loss: 1.2546
Epoch 15/30
172

In [ ]:
model.save_weights('model_rnn.weights.h5')
model2 = build_model(batch_size=1)
model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy())
model2.build(tf.TensorShape([1, None]))
model2.load_weights('model_rnn.weights.h5')
model2.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ (1, None, 256)         │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_8 (RNN)                     │ (1, None, 512)         │     1,182,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (1, None, 65)          │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,232,705 (4.70 MB)

 Trainable params: 1,232,705 (4.70 MB)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ (1, None, 256)         │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rnn_8 (RNN)                     │ (1, None, 512)         │     1,182,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (1, None, 65)          │        33,345 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,232,705 (4.70 MB)

 Trainable params: 1,232,705 (4.70 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
def generate_text(model, start_string, num_generate=500):
    # Converting start string to indices
    input_eval = [char2idx[s] for s in start_string]
    # Simulate a batch of 1 element
    input_eval = tf.expand_dims(input_eval, 0)
    # List to store generated text
    text_generated = []
    
    #Reset states of stateful layers (not the model)
    for layer in model.layers:
        if hasattr(layer, "reset_states"):
            layer.reset_states()
    
    for i in range(num_generate):
        # Probability prediction
        prediction = model(input_eval)
        
        # Take the last timestep
        prediction = prediction[:, -1, :]
        
        # Index prediction (argmax)
        index = tf.argmax(prediction, axis=-1).numpy()[0]
        
        # Feed back as next input
        input_eval = tf.expand_dims([index], 0)
        
        # Save character in generated list
        text_generated.append(idx2char[index])
    
    # Return the full generated sequence
    return start_string + ''.join(text_generated)

# Generate text
print(generate_text(model2, start_string="ROMEO:"))


ROMEO:
My lord, you have not seen the state of death
That thou hast spoke of him, for thou shalt not be a stool but this sentence of the people,
And then to be the world and mine own lord,
That we may come to the sun sets, which is no more strength.

KING RICHARD III:
So it is not so discover'd with the death of my consent,
And then to be the world and mine own lord,
That we may come to the sun sets, which is no more strength.

KING RICHARD III:
So it is not so discover'd with the death of my consent,


In [ ]:
'''
The code initially seeds with all given characters for the first prediction, then seeds with last one character at a time which is the previous prediction.
The result is good for a first try.
The model could be improved by adding more characters in the seed.
'''

## Part 2: RNN Embedding - Classification

In [151]:
import pandas as pd

df = pd.read_csv('MovieReview.csv')
df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})
print(df.head(5))

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1


/var/folders/y1/gws_3cyj7xx7gkncjr6_dxhm0000gn/T/ipykernel_27208/2812246460.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})


In [152]:
!pip install nltk
!pip install swifter
!pip install pandas

In [ ]:
import re
import unicodedata
import pandas as pd

try:
    import swifter
    use_swifter = True
except ModuleNotFoundError:
    use_swifter = False
    print("Swifter not installed. Using pandas apply instead.")

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, PunktSentenceTokenizer

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

stop_words = set(stopwords.words('english'))


def unicode_to_ascii(s):
    """Convert Unicode string to ASCII."""
    return ''.join(c for c in unicodedata.normalize('NFD', s)
                   if unicodedata.category(c) != 'Mn')

def preprocess_sentence(w):
    """Clean a sentence: lowercase, remove punctuation, stopwords, short words."""
    w = unicode_to_ascii(w.lower().strip())
    
    w = re.sub(r"([?.!,¿])", r" \1 ", w)
    
    w = re.sub(r'\s+', ' ', w)
    
    w = re.sub(r"[^a-zA-Z?.!]+", " ", w)
    
    w = re.sub(r'\b\w{0,2}\b', '', w)
    
    try:
        mots = word_tokenize(w.strip())
    except LookupError:
        tokenizer = PunktSentenceTokenizer()
        mots = []
        for sentence in tokenizer.tokenize(w.strip()):
            mots.extend(sentence.split())
    
    mots = [mot for mot in mots if mot not in stop_words]
    
    return ' '.join(mots).strip()


if use_swifter:
    df.review = df.review.swifter.apply(preprocess_sentence)
else:
    df.review = df.review.apply(preprocess_sentence)

df.head()


Pandas Apply:   0%|          | 0/50000 [00:00<?, ?it/s]

,review,sentiment
0,one reviewers mentioned watching episode hooke...,1
1,wonderful little production . filming techniqu...,1
2,thought wonderful way spend time hot summer we...,1
3,basically family little boy jake thinks zombie...,0
4,petter mattei love time money visually stunnin...,1


In [154]:
from sklearn.model_selection import train_test_split
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(df.review.values, df.sentiment.values, test_size=0.2, random_state=42)

In [155]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_text_train)

In [156]:
word2idx = tokenizer.word_index
idx2word = tokenizer.index_word
vocab_size = tokenizer.num_words

In [157]:
X_train = tokenizer.texts_to_sequences(X_text_train)
X_test = tokenizer.texts_to_sequences(X_text_test)

In [159]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_train = pad_sequences(X_train, maxlen=500, padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=500, padding='post', truncating='post')

In [160]:
modelMovie = tf.keras.Sequential([
    tf.keras.layers.Embedding(10000, 200, input_length=500),
    tf.keras.layers.RNN(tf.keras.layers.GRUCell(128), return_sequences=True),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(2, activation='softmax')
])

/opt/miniconda3/envs/venvMacOS/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [162]:
modelMovie.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=['accuracy'])
historyMovie = modelMovie.fit(X_train, y_text_train, validation_data=(X_test, y_text_test), epochs=3)

Epoch 1/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 420s 336ms/step - accuracy: 0.7637 - loss: 0.4360 - val_accuracy: 0.8969 - val_loss: 0.2539
Epoch 2/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 387s 310ms/step - accuracy: 0.9214 - loss: 0.2102 - val_accuracy: 0.8972 - val_loss: 0.2558
Epoch 3/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 383s 306ms/step - accuracy: 0.9516 - loss: 0.1352 - val_accuracy: 0.8878 - val_loss: 0.3476


In [163]:
test_loss, test_acc = modelMovie.evaluate(X_test, y_text_test)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}')

313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 79ms/step - accuracy: 0.8878 - loss: 0.3476
Test Loss: 0.3476, Test Accuracy: 0.8878


In [ ]:
'''
The model shows good performance 88% accuracy.
Preprocessing steps are crucial.
Summarizing a sequence in one vector is the key idea.
It will be interesting to try other architectures and compare performance.
We can observe that building complex models highly increases training time.
Because of that we have to particularly design and code our models to be efficient.
Designing carefully is the key to ensure flushing of time and resources.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Input: The product is terrible and broke immediately.
Predicted Sentiment: Negative with confidence 0.8390

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Input: Not bad, could be better.
Predicted Sentiment: Negative with confidence 0.9702

As you can see, the model doesn't correctly classify subtile meanings.
'''

In [164]:
inference_sequences = [
    "I loved this movie! It was amazing.",
    "The product is terrible and broke immediately.",
    "Not bad, could be better."
]

for seq in inference_sequences:
    seq_processed = preprocess_sentence(seq)
    seq_seq = tokenizer.texts_to_sequences([seq_processed])
    seq_padded = pad_sequences(seq_seq, maxlen=500, padding='post', truncating='post')
    prediction = modelMovie.predict(seq_padded)
    predicted_label = np.argmax(prediction)
    confidence = np.max(prediction)
    print(f"Input: {seq}\nPredicted Sentiment: {'Positive' if predicted_label == 1 else 'Negative'} with confidence {confidence:.4f}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
Input: I loved this movie! It was amazing.
Predicted Sentiment: Positive with confidence 0.9905

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
Input: The product is terrible and broke immediately.
Predicted Sentiment: Negative with confidence 0.8390

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Input: Not bad, could be better.
Predicted Sentiment: Negative with confidence 0.9702

